### Attention Map Cosine Sim

In [ ]:
import torch
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 모델 로딩
model = timm.create_model('vit_small_patch16_224', pretrained=True)
model.eval()

# 2. Attention map 저장용 리스트
attention_maps = []

# 3. Forward hook 함수 정의
def save_attention_map(module, input, output):
    # output shape: (B, num_heads, N, N)
    attention_maps.append(output.detach().cpu())

# Hook 등록 위치 수정
for block in model.blocks:
    block.attn.register_forward_hook(save_attention_map)  # <-- 수정됨

# 5. Dummy 입력 이미지
dummy_img = torch.randn(1, 3, 224, 224)  # 배치 1, RGB, 224x224

# 6. 모델 실행 (hook으로 attention 저장됨)
_ = model(dummy_img)

# attention flatten normalize
def flatten_and_normalize(attn_tensor):
    attn_mean = attn_tensor.mean(dim=0)
    flat = attn_mean.flatten()
    norm = flat.norm(p=2)
    if norm == 0:
        return flat
    return flat / norm

# 유사도 계산
n_layers = len(attention_maps)
if n_layers == 0:
    raise RuntimeError("Attention maps are empty. Check hook registration or model forward.")

similarity_matrix = torch.zeros(n_layers, n_layers)
for i in range(n_layers):
    attn_i = flatten_and_normalize(attention_maps[i][0])
    for j in range(n_layers):
        attn_j = flatten_and_normalize(attention_maps[j][0])
        similarity_matrix[i, j] = F.cosine_similarity(attn_i, attn_j, dim=0)

# NaN 확인
if torch.isnan(similarity_matrix).any():
    raise ValueError("Similarity matrix contains NaN values.")

# 9. 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(similarity_matrix.numpy(), cmap='viridis', annot=True, fmt=".2f")
plt.title("Layer-wise Attention Similarity (ViT-small)")
plt.xlabel("Layer")
plt.ylabel("Layer")
plt.show()


### Layer-wise CKA Similarity (MSA Output hidden state)

In [ ]:
import torch
import timm
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------------
# 1. 모델 로딩 및 Hook 세팅
# -------------------------------
model = timm.create_model('vit_base_patch16_224', pretrained=True)
model.eval()

msa_outputs = []

# hook에서 필요한 residual을 저장해줄 변수
for block in model.blocks:
    block.attn.input_residual = None  # residual x
    block.attn.drop_path = block.drop_path1  # DropPath 연결

# norm1 출력값을 통해 residual 저장
def save_residual(module, input, output):
    parent_block = None
    for block in model.blocks:
        if block.norm1 is module:
            parent_block = block
            break
    if parent_block is not None:
        parent_block.attn.input_residual = output

# attn module의 output을 받아서 최종 MSA output 계산
def save_msa_output(module, input, output):
    msa_out = module.input_residual + module.drop_path(output)
    msa_outputs.append(msa_out.detach().cpu())

# hook 등록
for block in model.blocks:
    block.norm1.register_forward_hook(save_residual)
    block.attn.register_forward_hook(save_msa_output)

# -------------------------------
# 2. Dummy 입력 (Batch > 1)
# -------------------------------
img = torch.randn(8, 3, 224, 224)  # Batch = 8
_ = model(img)

# -------------------------------
# 3. CKA 계산 함수
# -------------------------------
def gram_linear(x):
    return x @ x.T

def center_gram(K):
    n = K.size(0)
    unit = torch.ones(n, n, device=K.device) / n
    return K - unit @ K - K @ unit + unit @ K @ unit

def cka(X, Y):
    X = X - X.mean(dim=0)
    Y = Y - Y.mean(dim=0)

    K = center_gram(gram_linear(X))
    L = center_gram(gram_linear(Y))

    hsic = (K * L).sum()
    norm_K = torch.norm(K)
    norm_L = torch.norm(L)

    if norm_K == 0 or norm_L == 0:
        return torch.tensor(float("nan"))

    return hsic / (norm_K * norm_L)

# -------------------------------
# 4. Layer-wise CKA 계산 (MSA output 기반)
# -------------------------------
n_layers = len(msa_outputs)
cka_matrix = torch.zeros(n_layers, n_layers)

for i in range(n_layers):
    x_i = msa_outputs[i].reshape(-1, msa_outputs[i].shape[-1])  # [B * N, D]
    for j in range(n_layers):
        x_j = msa_outputs[j].reshape(-1, msa_outputs[j].shape[-1])
        cka_matrix[i, j] = cka(x_i, x_j)

cka_matrix = torch.nan_to_num(cka_matrix, nan=0.0)

# -------------------------------
# 5. 시각화
# -------------------------------
plt.figure(figsize=(10, 8))
sns.heatmap(
    cka_matrix.numpy(),
    cmap='rocket',       # 너가 원하는 색 조합
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    square=True,
    cbar=True
)
plt.title("Layer-wise CKA Similarity (MSA Output Hidden States)")
plt.xlabel("Layer")
plt.ylabel("Layer")
plt.tight_layout()
plt.show()



### Layer-wise CKA sim (Blokc output hidden state)

In [ ]:
import torch
import torch.nn.functional as F
import timm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------------
# 1. 모델 로드 및 Hook 설정
# -----------------------------------
model = timm.create_model('vit_base_patch16_224', pretrained=True)
model.eval()

features = []

# ViT 각 layer의 모든 토큰 hidden state 저장
def hook_fn(module, input, output):
    features.append(output.detach().cpu())  # shape: [B, N, D]

for block in model.blocks:
    block.register_forward_hook(hook_fn)

# -----------------------------------
# 2. Dummy 입력 (batch size > 1)
# -----------------------------------
img = torch.randn(8, 3, 224, 224)  # [B, C, H, W]
_ = model(img)  # Hook 실행됨

# -----------------------------------
# 3. CKA 함수 (flatten 후 비교)
# -----------------------------------
def gram_linear(x):
    return x @ x.T

def center_gram(K):
    n = K.size(0)
    unit = torch.ones(n, n, device=K.device) / n
    return K - unit @ K - K @ unit + unit @ K @ unit

def cka(X, Y):
    X = X - X.mean(dim=0)
    Y = Y - Y.mean(dim=0)

    K = center_gram(gram_linear(X))
    L = center_gram(gram_linear(Y))

    HSIC = (K * L).sum()
    norm_K = torch.norm(K)
    norm_L = torch.norm(L)

    if norm_K == 0 or norm_L == 0:
        return torch.tensor(float("nan"))

    return HSIC / (norm_K * norm_L)

# -----------------------------------
# 4. MSA 전체 representation → Flatten 후 CKA
# -----------------------------------
n_layers = len(features)
cka_matrix = torch.zeros(n_layers, n_layers)

for i in range(n_layers):
    x_i = features[i].reshape(-1, features[i].shape[-1])  # [B * N, D]
    for j in range(n_layers):
        x_j = features[j].reshape(-1, features[j].shape[-1])  # [B * N, D]
        cka_matrix[i, j] = cka(x_i, x_j)

cka_matrix = torch.nan_to_num(cka_matrix, nan=0.0)

# -----------------------------------
# 5. 시각화
# -----------------------------------
plt.figure(figsize=(10, 8))
sns.heatmap(
    cka_matrix.numpy(),
    cmap="rocket",       # 원하는 색감
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    square=True,
    cbar=True
)
plt.title("Layer-wise CKA Similarity (All Tokens)", fontsize=14)
plt.xlabel("Layer")
plt.ylabel("Layer")
plt.tight_layout()
plt.show()
